# When preserving granularity helps decisions

This notebook reproduces the mechanism and displays the full simulation's results.
Run `make granularity-run` at the repository root before executing it. The finite
population has known probabilities; these oracle metrics are not estimators from
ordinary held-out labels. No real-world welfare claim is made.

For score $S$, true risk $p=P(Y=1\mid S)$, forecast $Q=g(S)$, and
$r=E[p\mid Q]$, total probability error decomposes as
$E[(p-Q)^2]=E[(p-r)^2]+E[(r-Q)^2]$.

An action costing $t$ pays $Y$. Twice the uniformly integrated optimal value lost
from observing $Q$ instead of $S$ equals the first term. Twice the integrated
regret from acting iff $Q>t$ equals total error. Uniform threshold weighting is a
reference convention, not a measured distribution of costs.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():
    if root == root.parent:
        raise FileNotFoundError("Run inside the calibre repository")
    root = root.parent
sys.path.insert(0, str(root))

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display as show
from matplotlib_inline.backend_inline import set_matplotlib_formats

from experiments.granularity import report, study

%matplotlib inline
set_matplotlib_formats("svg")

arrays = report.load_results(study.ROOT / "results")
weights = report.bootstrap_weights()
print("Metric dimensions:", arrays["metrics"].shape)
print("Order: design, sample size, seed, method, precision, metric")

## Same calibration, different decision value

Both the fine risks and their pooled mean are calibrated. Pooling loss is $d^2$.
Noise added to a constant-risk forecast creates distinctions without information.

In [ ]:
for d in (0, 0.05, 0.15):
    p = np.array([0.3 - d, 0.3 + d])
    row = study.losses(p, np.full(2, 0.3))
    np.testing.assert_allclose(row["pooling_loss"], d * d, atol=1e-14)
    print(f"d={d:.2f}: pooling loss={row['pooling_loss']:.5f}")
with report.plt.rc_context(report.THEME):
    show(report.toy_figure())
report.plt.close("all")

## Fitted methods: information versus probability error

All methods share calibration observations. Evaluation is over the entire known
population, not a sample of outcomes. Blue is pooling loss and gray is
miscalibration; the sum is total probability error. Both exact and six-decimal
results are shown. An injective but wrong map has zero pooling loss.

In [ ]:
with report.plt.rc_context(report.THEME):
    for precision, label in enumerate(study.PRECISIONS):
        for d, title in enumerate(report.TITLES):
            print(title, label)
            show(report.decomposition_figure(arrays["metrics"], d, precision))
            report.plt.close("all")

## Threshold decisions

Differences in regret versus isotonic at calibration size 1,000; negative favors
the alternative. Bands are paired 95% pointwise bootstrap intervals for mean
performance across calibration samples, not ranges for individual fitted models.
No simultaneous dominance claim follows.

In [ ]:
with report.plt.rc_context(report.THEME):
    for precision in range(2):
        for d, title in enumerate(report.TITLES):
            print(title, study.PRECISIONS[precision])
            show(report.threshold_figure(arrays["curves"], d, precision, weights))
            report.plt.close("all")

## Capacity allocation and the tie-breaking comparator

Expected successes per 1,000 relative to original-score ranking. Positive is
better. Random ties are integrated exactly. Breaking ties with the original score
restores its ranking for nondecreasing maps. That ranking is useful in the
monotone-risk designs but can be harmful in the nonmonotone design.

In [ ]:
with report.plt.rc_context(report.THEME):
    for precision in range(2):
        print(study.PRECISIONS[precision])
        show(report.capacity_figure(arrays["capacity"], precision, weights))
        report.plt.close("all")

## Full-precision tables and limitations

The summary includes all designs, calibration sizes, methods, precision settings,
and metrics. Differences and intervals are paired against isotonic. Spearman is
missing when constant forecasts make it undefined. All threshold and capacity
curves, including sizes not plotted above, are in `evaluation.npz`.

These examples show a mechanism, not field evidence. Information recovery is
oracle recovery conditional on the fitted mapping. User knowledge, measurement
precision, costs, and whether the action affects the outcome can change the
practical conclusion. See DESIGN.md for the fixed design and source manifests for
provenance.

References: [Ehm et al. (2016)](https://arxiv.org/abs/1503.08195);
[Tasche (2021)](https://doi.org/10.1080/02331888.2021.2016767).

In [ ]:
summary = pd.read_csv(study.ROOT / "build" / "summary.csv")
show(summary)
print("Rows:", len(summary))